# Google Drive Torrent Downloader

This script allows downloading torrents directly to Google Drive through Google Colab.

⚠️ **Important**: For research and educational purposes only. Please respect copyright laws and use only for legal content.

## 1. Install Dependencies

In [1]:
# Install dependencies (fixed version)
!apt-get update
!apt-get install -y libtorrent-rasterbar-dev
!pip install libtorrent

# Check installation
import libtorrent as lt
print(f"✅ LibTorrent version: {lt.version}")
print("✅ Installation completed successfully!")

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [3,011 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates/restricted amd64 Packages [7,341 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/main amd64 Packages [4,292 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Get:12 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [10.2 MB]
Get:13 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,602 kB]
Hit:14

## 2. Mount Google Drive

In [3]:
from google.colab import drive
drive.mount('/content/drive')

# Create folder for torrents
import os
torrent_dir = '/content/drive/MyDrive/Torrents'
os.makedirs(torrent_dir, exist_ok=True)
print(f"Torrents folder: {torrent_dir}")

Mounted at /content/drive
Torrents folder: /content/drive/MyDrive/Torrents


## 3. Torrent Downloader

In [6]:
import libtorrent as lt
import time
import os
from IPython.display import clear_output

class TorrentDownloader:
    def __init__(self, download_path):
        self.download_path = download_path
        self.session = lt.session()
        self.session.listen_on(6881, 6891)

    def download_torrent(self, torrent_file_or_magnet):
        """Download torrent by file or magnet link with file selection"""
        try:
            # Check if it's a magnet link or file
            if torrent_file_or_magnet.startswith('magnet:'):
                handle = lt.add_magnet_uri(self.session, torrent_file_or_magnet, {
                    'save_path': self.download_path
                })
                print("Fetching metadata... (this may take a minute for magnet links)")
                # We must wait for metadata to know what files are inside
                while not handle.has_metadata():
                    time.sleep(1)
            else:
                # Load torrent file
                info = lt.torrent_info(torrent_file_or_magnet)
                handle = self.session.add_torrent({
                    'ti': info,
                    'save_path': self.download_path
                })

            # --- FILE SELECTION LOGIC ---
            info = handle.get_torrent_info()
            num_files = info.num_files()

            print(f"\nTorrent name: {handle.name()}")
            print("\n--- Files inside this torrent ---")

            for i in range(num_files):
                file_size_mb = info.files().file_size(i) / (1024 * 1024)
                print(f"[{i}] {info.files().file_path(i)} ({file_size_mb:.2f} MB)")

            print("\n---------------------------------")
            choice = input("Enter the file numbers you want to download separated by commas (e.g. 0,2,5) OR press Enter to download everything: ")

            if choice.strip():
                # Parse the numbers typed in
                selected_indexes = [int(x.strip()) for x in choice.split(",") if x.strip().isdigit()]

                # Create a list where all files are initially set to 0 priority (Skip)
                priorities = [0] * num_files

                # Change priority of selected files to 1 (Download)
                for idx in selected_indexes:
                    if 0 <= idx < num_files:
                        priorities[idx] = 1

                handle.prioritize_files(priorities)
                print(f"\nSetting updated! Only downloading {len(selected_indexes)} selected file(s).")
            else:
                print("\nNo specific files selected. Downloading the entire torrent.")

            time.sleep(2) # Brief pause to read the output before screen clears
            # ----------------------------

            # Progress monitoring
            while not handle.is_seed():
                status = handle.status()

                clear_output(wait=True)
                print(f"Torrent: {handle.name()}")
                print(f"Progress: {status.progress * 100:.1f}%")
                print(f"Download rate: {status.download_rate / 1000:.1f} KB/s")
                print(f"Upload rate: {status.upload_rate / 1000:.1f} KB/s")
                print(f"Peers: {status.num_peers}")
                print(f"Seeds: {status.num_seeds}")

                if status.state == lt.torrent_status.downloading:
                    print("Status: Downloading")
                elif status.state == lt.torrent_status.finished:
                    print("Status: Finished")
                    break
                elif status.state == lt.torrent_status.seeding:
                    print("Status: Seeding")
                    break

                time.sleep(2)

            print(f"\n✅ Download completed: {handle.name()}")
            print(f"Files saved to: {self.download_path}")

        except Exception as e:
            print(f"❌ Download error: {e}")

    def list_files(self):
        """Show downloaded files"""
        print("\n📁 Downloaded files:")
        for root, dirs, files in os.walk(self.download_path):
            level = root.replace(self.download_path, '').count(os.sep)
            indent = ' ' * 2 * level
            print(f"{indent}{os.path.basename(root)}/")
            sub_indent = ' ' * 2 * (level + 1)
            for file in files:
                file_path = os.path.join(root, file)
                size = os.path.getsize(file_path)
                size_mb = size / (1024 * 1024)
                print(f"{sub_indent}{file} ({size_mb:.1f} MB)")

# Create downloader instance
downloader = TorrentDownloader(torrent_dir)
print("✅ Torrent downloader ready with file selection enabled!")

✅ Torrent downloader ready with file selection enabled!


/tmp/ipykernel_3227/827194187.py:10: DeprecationWarning: listen_on() is deprecated
  self.session.listen_on(6881, 6891)


## 4. Usage

In [ ]:
# Option 1: Download by magnet link
magnet_link = "magnet:?xt=urn:btih:F0DDB30B9416B26AC4AFC0AF86BDAA304A40DAEE&dn=Six+Feet+Under+%282001%29+Season+1-5+S01-S05+%281080p+AMZN+WEB-DL+x265+HEVC+10bit+EAC3+5.1+Silence%29+REPACK+%5BQxR%5D+&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337%2Fannounce&tr=udp%3A%2F%2Ftracker.openbittorrent.com%3A6969%2Fannounce&tr=udp%3A%2F%2Fexodus.desync.com%3A6969%2Fannounce&tr=udp%3A%2F%2Fwww.torrent.eu.org%3A451%2Fannounce&tr=udp%3A%2F%2Ftracker.torrent.eu.org%3A451%2Fannounce&tr=udp%3A%2F%2Fopen.tracker.cl%3A1337%2Fannounce&tr=udp%3A%2F%2Fexplodie.org%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.opentrackr.org%3A1337%2Fannounce&tr=http%3A%2F%2Ftracker.openbittorrent.com%3A80%2Fannounce&tr=udp%3A%2F%2Fopentracker.i2p.rocks%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.internetwarriors.net%3A1337%2Fannounce&tr=udp%3A%2F%2Ftracker.leechers-paradise.org%3A6969%2Fannounce&tr=udp%3A%2F%2Fcoppersurfer.tk%3A6969%2Fannounce&tr=udp%3A%2F%2Ftracker.zer0day.to%3A1337%2Fannounce"
downloader.download_torrent(magnet_link)

# Option 2: Download by torrent file
# First upload torrent file to Colab
from google.colab import files
print("Select torrent file for download:")
uploaded = files.upload()

# Get uploaded file name
torrent_file = list(uploaded.keys())[0]
print(f"Uploaded file: {torrent_file}")

# Start download
downloader.download_torrent(torrent_file)

Torrent: Six Feet Under (2001) Season 1-5 S01-S05 (1080p AMZN WEB-DL x265 HEVC 10bit EAC3 5.1 Silence) REPACK
Progress: 21.4%
Download rate: 674.1 KB/s
Upload rate: 29.4 KB/s
Peers: 6
Seeds: 5
Status: Downloading


## 5. View Downloaded Files

In [ ]:
# Show all downloaded files
downloader.list_files()

## 6. Manual Download (Alternative Method)

In [ ]:
# For direct usage without class
def download_torrent_simple(magnet_or_file, save_path):
    import libtorrent as lt

    session = lt.session()
    session.listen_on(6881, 6891)

    if magnet_or_file.startswith('magnet:'):
        handle = lt.add_magnet_uri(session, magnet_or_file, {'save_path': save_path})
    else:
        info = lt.torrent_info(magnet_or_file)
        handle = session.add_torrent({'ti': info, 'save_path': save_path})

    print(f"Downloading: {handle.name()}")

    while not handle.is_seed():
        status = handle.status()
        print(f"\rProgress: {status.progress * 100:.1f}%", end='', flush=True)

        if status.state == lt.torrent_status.finished:
            break

        time.sleep(1)

    print(f"\n✅ Download completed!")

# Usage example:
# download_torrent_simple("magnet:?xt=urn:btih:HASH", "/content/drive/MyDrive/Torrents")

## 📝 Notes

- Files will be saved to `/content/drive/MyDrive/Torrents/` folder on your Google Drive
- Colab session may disconnect after 12 hours of inactivity
- For large files, it's recommended to check progress regularly
- **For research and educational purposes only**
- Use only for legal content and respect copyright laws
- If you encounter connection issues, try restarting the dependencies installation cell